<a href="https://colab.research.google.com/github/TaiwoOlaniyiToheeb/naive_bayes/blob/main/naive_bays.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/colab files/nba.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
bat_mat = df.corr()
bat_mat

In [ ]:
plt.figure(figsize = (10, 5))
sns.heatmap(bat_mat, annot = True, annot_kws = {'size' : 8}, linewidths = .6, linecolor = 'white')
plt.show()

### Target Value

In [ ]:
df['target_5yrs'].value_counts()

In [ ]:
df['target_5yrs'].value_counts(normalize = True) * 100

There are 831, 1 values in the target column accounting 62.01% and 509, 0 values accounting for 37.99%. This is a fairly balanced target values. This is not major imbalance and is ready for machine learning.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, f1_score, precision_score, recall_score

In [ ]:
X = df.drop(['target_5yrs'], axis = 1)
y = df['target_5yrs']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)

In [ ]:
print("Number of mislabeled points out of a total %d points : %d"% (X_test.shape[0], (y_test != y_pred).sum()))

In [ ]:
print(f'Accuracy Score : {accuracy_score(y_test, y_pred)}')
print(f'Precision Score : {precision_score(y_test, y_pred)}')
print(f'F1 Score : {f1_score(y_test, y_pred)}')
print(f'Confusion Matrix : \n{confusion_matrix(y_test, y_pred)}')

In [ ]:
print(f"Recall Score: {recall_score(y_test, y_pred)}")

In [ ]:
o_fit =  confusion_matrix(y_test, y_pred)
sns.heatmap(o_fit, annot = True, fmt = 'd')
plt.show()

There are 90 mis-classified profiles. We will run tunning

In [ ]:
cl_rep_dict = classification_report(y_test, y_pred, output_dict=True)
cl = pd.DataFrame(cl_rep_dict).transpose()
cl

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    'var_smoothing': np.logspace(0,-9, num=100)
}
gsv = GridSearchCV(estimator = GaussianNB(), param_grid = param_grid, verbose = 1, cv = 5, n_jobs = -1)
gsv.fit(X_train, y_train)
print(gsv.best_estimator_)

In [ ]:
y_pred = gsv.predict(X_test)


In [ ]:
print(f'Accuracy Score : {accuracy_score(y_test, y_pred)}')

In [ ]:
con_ma = confusion_matrix(y_test, y_pred)
con_ma

In [ ]:
sns.heatmap(con_ma, annot = True, fmt = 'd')
plt.show()

In [ ]:
print(f"Recall Score: {recall_score(y_test, y_pred)}")

In [ ]:
print(y_test.value_counts())
print(y_pred)

In [ ]:
print(pd.Series(y_pred).value_counts())

In [ ]:
print(f'Precision Score : {precision_score(y_test, y_pred)}')
print(f'F1 Score : {f1_score(y_test, y_pred)}')

In [ ]:
pd.DataFrame({
    'Feature': X.columns,
    'Class_0_Mean': gsv.best_estimator_.theta_[0],
    'Class_1_Mean': gsv.best_estimator_.theta_[1]
}).head()

Hyperparameter tuning improved the Gaussian Naive Bayes model's ability to identify players likely to remain in the league for at least five years. Accuracy increased from 66.4% to 69.0%, recall improved substantially from 55.0% to 79.3%, and F1-score increased from 0.674 to 0.764. However, precision decreased from 86.9% to 73.6%, indicating a higher number of false positives. This reflects the inherent tradeoff between precision and recall. Depending on whether the objective is to avoid drafting busts or avoid missing talented players, either model may be preferred.

## Naive Bayes Independence Assumption

Naive Bayes is called "naive" because it assumes that all predictor variables are **conditionally independent** given the target class. In other words, once we know whether a player will remain in the league for at least five years (`target_5yrs`), the model assumes that one feature does not influence another.

For example, the model assumes:

* Points scored is independent of minutes played.
* Rebounds are independent of games played.
* Assists are independent of field goals made.

Mathematically, the probability of a player belonging to a class is computed as:

[
P(X|Y) = P(x_1|Y)P(x_2|Y)\cdots P(x_n|Y)
]

where the features are treated as independent of one another.

However, this assumption is often unrealistic in basketball data because many player statistics are naturally correlated.

### Examples of correlated basketball statistics

* **Points and minutes played:** Players who play more minutes generally score more points.
* **Field goals made and points:** More made shots directly increase points scored.
* **Rebounds and minutes played:** Players on the court longer usually collect more rebounds.
* **Assists and usage rate:** Players with greater offensive roles tend to record more assists.

Because these relationships exist, the independence assumption is violated in practice.

Despite this limitation, Naive Bayes often performs surprisingly well because it focuses on classification accuracy rather than perfectly modeling the underlying relationships between variables.

---

## Model Reliability for a Scouting Department

The Gaussian Naive Bayes model provides a useful data-driven tool for identifying players likely to remain in the NBA for at least five years. Hyperparameter tuning improved the model's ability to detect future long-term players by substantially increasing recall.

### Strengths

* Fast and computationally efficient.
* Easy to interpret and implement.
* Performs well even with relatively small datasets.
* Successfully identifies a large proportion of future long-term players.
* Can serve as an objective supplement to traditional scouting.

### Limitations

* Assumes independence among player statistics, which is unrealistic in basketball.
* May oversimplify complex relationships between player attributes.
* Sensitive to the assumption that features follow a Gaussian distribution.
* Cannot capture nonlinear interactions between variables.
* Model predictions should not replace expert scouting evaluations.

---

